<a href="https://colab.research.google.com/github/demichie/Principles-of-Numerical-Modelling-in-Geosciences/blob/main/Chapter8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 8: The Advection-Diffusion Equation

This notebook contains the Python code examples for Chapter 8 of "Principles of Numerical Modelling in Geosciences".

This chapter integrates the concepts from the previous two by tackling the advection-diffusion equation, a PDE that models a vast range of real-world geoscience phenomena where transport and spreading occur simultaneously. We will develop a combined numerical scheme, explore its stability, and use it to simulate contaminant transport in an aquifer.

## 8.5 Python Implementation: Contaminant Transport in an Aquifer

To see the advection-diffusion equation in action, we will simulate a practical geoscience problem: the one-dimensional transport of a conservative solute (a contaminant) in a uniform, saturated aquifer.

### Problem Setup

We consider the following idealized scenario:
- **Domain:** A 1D horizontal aquifer of length $L=200$ m.
- **Flow and Transport Properties:** A steady, uniform groundwater flow with velocity $u=0.5$ m/day. The contaminant has a longitudinal dispersion coefficient of $D$, which we will vary.
- **Initial Condition (t=0):** The aquifer is initially clean ($\phi=0$), except for a localized slug of contaminant represented by a Gaussian pulse centred at $x_0 = 40$ m with an initial standard deviation $\sigma_0 = 8.0$ m.
- **Boundary Conditions (for t>0):** A Dirichlet inflow ($\phi(0, t) = 0$) and a Neumann (Danckwerts) outflow ($\frac{\partial \phi}{\partial x}(L, t) = 0$).
- **Simulation Time:** We will track the plume for $t_{final}=20$ days.

To analyse the system's behaviour, we will use the **Péclet number**. We will fix a characteristic length scale for our problem, $L_{char} = 10$ m, and use the Péclet number, $Pe = u L_{char} / D$, to classify the transport regime.

### Python Code and Results

The Python script below implements the explicit upwind-centered scheme to solve our contaminant transport problem. To facilitate the exploration of different physical regimes, the core of the numerical solver is encapsulated within a dedicated function, `run_advection_diffusion()`.

This function takes the diffusion coefficient `D_coeff` as a primary input and performs all the necessary steps for a complete simulation:
1. It calculates the maximum stable time step, $\Delta t$, by considering both the advective and diffusive stability constraints.
2. It sets up the computational grid and the initial Gaussian profile.
3. It executes the main time-stepping loop, applying the explicit update formula and enforcing the boundary conditions at each step.
4. Finally, it returns the results of the simulation, including the final concentration profile and the calculated Péclet number.

This modular structure allows us to easily call the same solver three times with different values of `D_coeff` to generate the solutions for our advection-dominated, mixed, and diffusion-dominated cases.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Simulation Parameters (Global) ---
lengthDomain = 200.0   # m (Extended domain to avoid boundary effects)
advectionVelocity = 0.5 # m/day
numXPoints = 401        # Increased to maintain good resolution
dxStep = lengthDomain / (numXPoints - 1)
timeFinal = 20.0 # days

# --- 2. Function for Initial Condition ---
def initialConditionGaussian(x, x0, sigma):
    """Creates a Gaussian pulse profile."""
    return np.exp(-0.5 * ((x - x0) / sigma)**2)

# --- 3. Main simulation function to run for different D ---
def run_advection_diffusion(D_coeff, safety_factor=0.5):
    """
    Runs a 1D advection-diffusion simulation for a given diffusion coefficient.
    """
    # Calculate the stable time step based on both constraints
    dt_adv_limit = dxStep / advectionVelocity
    dt_diff_limit = 0.5 * dxStep**2 / (D_coeff + 1e-9)
    dtStep = safety_factor * min(dt_adv_limit, dt_diff_limit)
    numTimeSteps = int(timeFinal / dtStep)

    # Peclet number based on a fixed characteristic
    # length L_char = 10 m
    L_char = 10.0
    peclet = advectionVelocity * L_char / (D_coeff + 1e-9)

    print(f"\n--- Running for D = {D_coeff:.3f} m^2/day (Pe = {peclet:.2f}) ---")

    # Grid and Initialization
    xGrid = np.linspace(0, lengthDomain, numXPoints)
    phiInitial = initialConditionGaussian(xGrid, x0=40.0, sigma=8.0)
    phiCurrent = phiInitial.copy()

    # --- Time-stepping Loop ---
    for n in range(numTimeSteps):
        phiOld = phiCurrent.copy()

        # Update interior points
        for i in range(1, numXPoints - 1):
            adv_term = advectionVelocity * (phiOld[i] - phiOld[i-1]) / dxStep
            diff_term = D_coeff * (phiOld[i+1] - 2*phiOld[i] + phiOld[i-1]) / dxStep**2
            phiCurrent[i] = phiOld[i] - dtStep * adv_term + dtStep * diff_term

        # Apply Boundary Conditions
        phiCurrent[0] = 0.0
        phiCurrent[-1] = phiCurrent[-2]

    return xGrid, phiInitial, phiCurrent, peclet

# --- 5. Run Simulations for the Three Distinct Regimes ---
L_char = 10.0

# Case 1: Advection-Dominated (Target Pe = 25.0)
D1 = (advectionVelocity * L_char) / 25.0 # D = 0.5 * 10 / 25.0 = 0.2
x1, phi_initial1, phi_final1, pe1 = run_advection_diffusion(D1)

# Case 2: Mixed Regime (Target Pe = 1.25)
D2 = (advectionVelocity * L_char) / 1.25 # D = 0.5 * 10 / 1.25 = 4.0
x2, phi_initial2, phi_final2, pe2 = run_advection_diffusion(D2)

# Case 3: Diffusion-Dominated (Target Pe = 0.05)
D3 = (advectionVelocity * L_char) / 0.05 # D = 0.5 * 10 / 0.95 = 100.0
x3, phi_initial3, phi_final3, pe3 = run_advection_diffusion(D3)

# --- 6. Plotting Results ---
fig, axs = plt.subplots(3, 1, figsize=(10, 8), sharex=True, sharey=True)
fig.supxlabel('Position x (m)', fontsize=12) # Shared x-axis label at the bottom

# Plot 1: Advection-Dominated
axs[0].plot(x1, phi_initial1, 'k:', label='Initial Condition (t=0)')
axs[0].plot(x1, phi_final1, 'b-', lw=2, label=f'Final State (t={timeFinal:.0f} days)')
axs[0].set_title(f'Advection-Dominated (Pe = {pe1:.1f})')
axs[0].set_ylabel('Concentration $\\phi$')
axs[0].legend()
axs[0].grid(True)
axs[0].set_xlim(0, lengthDomain)
axs[0].set_ylim(-0.05, 1.05)

# Plot 2: Mixed Regime (Green plot)
axs[1].plot(x2, phi_initial2, 'k:', label='Initial Condition (t=0)')
axs[1].plot(x2, phi_final2, 'g-', lw=2, label=f'Final State (t={timeFinal:.0f} days)')
axs[1].set_title(f'Mixed Regime (Pe = {pe2:.2f})')
axs[1].set_ylabel('Concentration $\\phi$')
axs[1].legend()
axs[1].grid(True)

# Plot 3: Diffusion-Dominated (Red plot)
axs[2].plot(x3, phi_initial3, 'k:', label='Initial Condition (t=0)')
axs[2].plot(x3, phi_final3, 'r-', lw=2, label=f'Final State (t={timeFinal:.0f} days)')
axs[2].set_title(f'Diffusion-Dominated (Pe = {pe3:.2f})')
axs[2].set_ylabel('Concentration $\\phi$')
axs[2].legend()
axs[2].grid(True)

plt.tight_layout()
plt.show()

## Chapter 8 Exercises

Test your understanding of the concepts covered in this chapter by solving the following problems.

### E8.1: Stability Constraints in Practice

Consider the Python solver from Listing 8.1. Suppose you are modelling a system with the following physical parameters:
- Advection velocity $u = 0.2$ m/s
- Diffusion coefficient $D = 0.001$ m$^2$/s
- Domain length $L = 50$ m
- Number of grid points $N_x = 251$

1. Calculate the spatial step size $\Delta x$.
2. Calculate the maximum allowable time step $\Delta t_{adv}$ based only on the advective CFL condition ($C \le 1$).
3. Calculate the maximum allowable time step $\Delta t_{diff}$ based only on the diffusive stability condition ($\alpha \le 0.5$).
4. What is the actual maximum time step $\Delta t_{max}$ you can use for this simulation to be stable? Which process is limiting the time step?
5. If you were to double the spatial resolution (i.e., halve $\Delta x$), by what factor would $\Delta t_{max}$ have to decrease?

In [ ]:
# E8.1: Solution
u = 0.2  # m/s
D = 0.001 # m^2/s
L = 50.0  # m
Nx = 251

# 1. Calculate spatial step size
dx = L / (Nx - 1)
print(f"1. Spatial step size dx = {dx:.3f} m")

# 2. Calculate advective time step limit
dt_adv = dx / abs(u)
print(f"2. Advective stability limit (C<=1): dt <= {dt_adv:.3f} s")

# 3. Calculate diffusive time step limit
dt_diff = 0.5 * dx**2 / D
print(f"3. Diffusive stability limit (alpha<=0.5): dt <= {dt_diff:.3f} s")

# 4. Determine the actual maximum time step
dt_max = min(dt_adv, dt_diff)
print(f"\n4. The actual maximum stable time step is dt_max = {dt_max:.3f} s.")
if dt_adv < dt_diff:
    print("   The advective process (CFL condition) is limiting the time step.")
else:
    print("   The diffusive process is limiting the time step.")

# 5. Effect of halving dx
dx_new = dx / 2
dt_adv_new = dx_new / abs(u)
dt_diff_new = 0.5 * dx_new**2 / D
dt_max_new = min(dt_adv_new, dt_diff_new)

factor_change = dt_max / dt_max_new

print(f"\n5. After halving dx to {dx_new:.3f} m:")
print(f"   New advective limit: dt <= {dt_adv_new:.3f} s (a factor of 2 smaller)")
print(f"   New diffusive limit: dt <= {dt_diff_new:.3f} s (a factor of 4 smaller)")
print(f"   The new dt_max is {dt_max_new:.3f} s.")
print(f"   The maximum time step had to be decreased by a factor of {factor_change:.1f}.")

### E8.2: Exploring the Péclet Number

Using the script from Listing 8.1 as a basis, perform a numerical experiment. Keep the advection velocity constant at $u=0.5$ m/day and the domain length $L=200$ m.

1. Run three simulations up to $t_{final}=100$ days with the following diffusion coefficients:
   - $D = 0.1$ m$^2$/day
   - $D = 2.5$ m$^2$/day
   - $D = 50.0$ m$^2$/day
2. For each case, calculate the Péclet number ($Pe = uL/D$).
3. Plot the final concentration profile for all three cases on the same graph, along with the initial condition. Use different colours for each case and include a legend that lists the Péclet number.
4. Describe how the shape and position of the final pulse change as the Péclet number decreases. How does this relate to the relative dominance of advection and diffusion?



In [ ]:
# E8.2: Exploring the Péclet Number
import numpy as np
import matplotlib.pyplot as plt

def initialConditionGaussian(x, x0, sigma):
    """Creates a Gaussian pulse profile."""
    return np.exp(-0.5 * ((x - x0) / sigma)**2)

# We can reuse the run_advection_diffusion function from the chapter's main script
# but we need to modify it to take L and t_final as arguments
def run_adv_diff_exercise(L, t_final, D_coeff, u=0.5, Nx=201, safety_factor=0.5):
    dx = L / (Nx - 1)
    dt_adv = dx / u
    dt_diff = 0.5 * dx**2 / (D_coeff + 1e-9)
    dt = safety_factor * min(dt_adv, dt_diff)
    Nt = int(t_final / dt)

    peclet = u * L / (D_coeff + 1e-9)
    print(f"Running for D={D_coeff:.1f}, Pe={peclet:.1f}")

    x = np.linspace(0, L, Nx)
    phi_initial = initialConditionGaussian(x, x0=L*0.2, sigma=L*0.08)
    phi_current = phi_initial.copy()

    for _ in range(Nt):
        phi_old = phi_current.copy()
        for i in range(1, Nx - 1):
            adv = u * (phi_old[i] - phi_old[i-1]) / dx
            diff = D_coeff * (phi_old[i+1] - 2*phi_old[i] + phi_old[i-1]) / dx**2
            phi_current[i] = phi_old[i] - dt*adv + dt*diff
        phi_current[0] = 0
        phi_current[-1] = phi_current[-2]

    return x, phi_initial, phi_current, peclet

# Parameters for the exercise
L_ex = 200.0
t_final_ex = 100.0
D_values = [0.1, 2.5, 50.0]

plt.figure(figsize=(10, 6))

# Run and plot for each D
x_ref, phi_ref, _, _ = run_adv_diff_exercise(L_ex, t_final_ex, D_values[0])
plt.plot(x_ref, phi_ref, 'k:', label='Initial Condition')

colors = ['b', 'g', 'r']
for i, D_val in enumerate(D_values):
    x, _, phi_final, pe = run_adv_diff_exercise(L_ex, t_final_ex, D_val)
    plt.plot(x, phi_final, color=colors[i], lw=2, label=f'D={D_val:.1f}, Pe={pe:.1f}')

plt.title('E8.2: Exploring the Péclet Number')
plt.xlabel('Position x (m)')
plt.ylabel('Concentration $\\phi$')
plt.legend()
plt.grid(True)
plt.show()

### E8.3: The Mistake of Using FTCS for Advection (Conceptual)

*This is a conceptual exercise with no coding required. The answers are derived from the definitions and discussions in the chapter.*